In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv
/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv


In [2]:
movies_path = "/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv"
credits_path = "/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv"

movies_df = pd.read_csv(movies_path)
credits_df = pd.read_csv(credits_path)


In [3]:
movies_df.info()
credits_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [4]:
movies_df.head()
credits_df.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## 2.1 Análisis Inicial

Se utilizó `.info()` para identificar los tipos de datos y `.head()` para observar ejemplos de los valores.

Se observó que varias columnas tienen tipo `object` pero no todas contienen JSON, por lo que se analizaron manualmente.

Columnas con valores múltiples en formato JSON:

En movies_df:
- genres
- keywords
- production_companies
- production_countries
- spoken_languages

En credits_df:
- cast
- crew

Estas columnas contienen listas de diccionarios dentro de una sola celda, lo cual representa múltiples valores en un solo atributo.

Esto viola la atomicidad requerida en bases de datos relacionales.

### Relación con 1NF

La Primera Forma Normal (1NF) establece que cada celda debe contener un valor atómico.

Las columnas identificadas no cumplen con 1NF porque contienen listas de valores (JSON) en una sola celda.

Por lo tanto, es necesario separar estos datos en tablas independientes para eliminar los grupos repetitivos.

## 2.2 Diagrama Entidad-Relación

![ER Diagram](https://i.imgur.com/1BkjzOT.png)

El diagrama E/R representa las entidades principales del sistema, como Movie, Actor, Genre, Company, Country, Language y Crew.

Se identificaron relaciones muchos-a-muchos entre Movie y las demás entidades debido a que una película puede tener múltiples actores, géneros, compañías, etc.

Estas relaciones serán resueltas en el modelo relacional mediante tablas intermedias.

## 2.3 Modelo Relacional y Normalización

Luego de identificar las columnas JSON, se tradujo el modelo E/R a un modelo relacional. La idea principal es separar los datos repetidos en tablas diferentes para que cada tabla represente una sola entidad o relación.

### Esquema lógico inicial

Movie(
    movie_id PK,
    title,
    budget,
    homepage,
    original_language,
    original_title,
    overview,
    popularity,
    release_date,
    revenue,
    runtime,
    status,
    tagline,
    vote_average,
    vote_count
)

Genre(
    genre_id PK,
    genre_name
)

Movie_Genre(
    movie_id FK,
    genre_id FK,
    PK(movie_id, genre_id)
)

Keyword(
    keyword_id PK,
    keyword_name
)

Movie_Keyword(
    movie_id FK,
    keyword_id FK,
    PK(movie_id, keyword_id)
)

Production_Company(
    company_id PK,
    company_name
)

Movie_Production_Company(
    movie_id FK,
    company_id FK,
    PK(movie_id, company_id)
)

Production_Country(
    country_code PK,
    country_name
)

Movie_Production_Country(
    movie_id FK,
    country_code FK,
    PK(movie_id, country_code)
)

Spoken_Language(
    language_code PK,
    language_name
)

Movie_Spoken_Language(
    movie_id FK,
    language_code FK,
    PK(movie_id, language_code)
)

Actor(
    actor_id PK,
    actor_name,
    gender
)

Movie_Actor(
    movie_id FK,
    actor_id FK,
    character,
    cast_order,
    PK(movie_id, actor_id, character)
)

Crew_Member(
    crew_id PK,
    crew_name,
    gender
)

Movie_Crew(
    movie_id FK,
    crew_id FK,
    department,
    job,
    PK(movie_id, crew_id, department, job)
)

### 1NF - Primera Forma Normal

Para cumplir con 1NF, cada celda debe contener un solo valor atómico. En el dataset original, varias columnas no cumplen esto porque contienen listas JSON.

Columnas con valores múltiples:
- genres
- keywords
- production_companies
- production_countries
- spoken_languages
- cast
- crew

Ejemplo:
La columna `cast` contiene varios actores dentro de una sola celda. Cada actor tiene datos como `cast_id`, `character`, `name` y `gender`.

Esto se corrigió separando esas listas en tablas nuevas como `Actor` y `Movie_Actor`.

Así se eliminan los grupos repetitivos y cada fila representa una sola relación.

### 2NF - Segunda Forma Normal

Para cumplir con 2NF, se eliminaron dependencias parciales.

En las tablas intermedias se usan llaves compuestas, por ejemplo:

Movie_Actor(movie_id, actor_id)

Si se dejara `actor_name` dentro de `Movie_Actor`, habría una dependencia parcial porque `actor_name` depende solo de `actor_id`, no de toda la llave compuesta.

Por eso se separó así:

Actor(actor_id, actor_name, gender)

Movie_Actor(movie_id, actor_id, character, cast_order)

Lo mismo se aplicó para géneros, keywords, compañías, países e idiomas. Los nombres se guardan en tablas maestras/catálogos y las relaciones con películas se guardan en tablas intermedias.

### 3NF - Tercera Forma Normal

Para cumplir con 3NF, se eliminaron dependencias transitivas. Esto significa que los atributos no clave no deben depender de otro atributo no clave.

Ejemplo:
En vez de guardar `country_name` directamente en la película, se crea una tabla separada:

Production_Country(country_code, country_name)

Luego la película se relaciona con el país usando:

Movie_Production_Country(movie_id, country_code)

De esta forma, `country_name` depende de `country_code`, no directamente de `movie_id`.

También se aplicó la misma idea con:
- genre_name depende de genre_id
- keyword_name depende de keyword_id
- company_name depende de company_id
- language_name depende de language_code
- actor_name depende de actor_id
- crew_name depende de crew_id

### BCNF / 4NF

Para BCNF y 4NF, se identificaron dependencias multivaluadas independientes.

Una película puede tener:
- muchos géneros
- muchos actores
- muchas compañías productoras
- muchos países de producción
- muchos idiomas
- muchos miembros del crew

Estas listas son independientes entre sí. Por ejemplo, los actores de una película no dependen de sus géneros. Si se dejaran todos juntos en una sola tabla, se crearían muchas repeticiones innecesarias.

Por eso se separaron en tablas intermedias:

- Movie_Genre
- Movie_Keyword
- Movie_Production_Company
- Movie_Production_Country
- Movie_Spoken_Language
- Movie_Actor
- Movie_Crew

Esto evita redundancia y hace que el diseño sea más limpio.

## 2.4 Entrega Visual

En esta sección se presentan los diagramas desarrollados para el modelado conceptual y lógico de la base de datos.

### Diagrama Entidad-Relación (E/R)

El siguiente diagrama representa el modelo conceptual, donde se identifican las entidades principales y sus relaciones (muchos a muchos) derivadas del análisis del dataset.

![ER Diagram](https://i.imgur.com/1BkjzOT.png)

---

### Modelo Relacional Final

El siguiente diagrama muestra el modelo lógico resultante luego del proceso de normalización (1NF, 2NF, 3NF y 4NF).

Se incluyen:
- Tablas principales (Movie, Actor, Genre, etc.)
- Tablas intermedias para resolver relaciones muchos-a-muchos
- Llaves primarias (PK) y foráneas (FK)

![Relational Model](https://i.imgur.com/Xp6d8ZB.png)

## 3.1 ETL Process

En este paso, extraemos los datos de los archivos CSV y transformamos las columnas JSON en tablas normalizadas.

In [5]:
import pandas as pd
import json
import sqlite3

In [6]:
movies_path = "/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv"
credits_path = "/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv"

movies_df = pd.read_csv(movies_path)
credits_df = pd.read_csv(credits_path)

movies_df.head(), credits_df.head()

(      budget                                             genres  \
 0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
 1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
 2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
 3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
 4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
 
                                        homepage      id  \
 0                   http://www.avatarmovie.com/   19995   
 1  http://disney.go.com/disneypictures/pirates/     285   
 2   http://www.sonypictures.com/movies/spectre/  206647   
 3            http://www.thedarkknightrises.com/   49026   
 4          http://movies.disney.com/john-carter   49529   
 
                                             keywords original_language  \
 0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
 1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en   
 2 

In [7]:
# Rename id to movie_id so both files use the same key name
movies_df = movies_df.rename(columns={"id": "movie_id"})

movie_table = movies_df[[
    "movie_id",
    "title",
    "budget",
    "homepage",
    "original_language",
    "original_title",
    "overview",
    "popularity",
    "release_date",
    "revenue",
    "runtime",
    "status",
    "tagline",
    "vote_average",
    "vote_count"
]].copy()

movie_table.head()

,movie_id,title,budget,homepage,original_language,original_title,overview,popularity,release_date,revenue,runtime,status,tagline,vote_average,vote_count
0,19995,Avatar,237000000,http://www.avatarmovie.com/,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,2009-12-10,2787965087,162.0,Released,Enter the World of Pandora.,7.2,11800
1,285,Pirates of the Caribbean: At World's End,300000000,http://disney.go.com/disneypictures/pirates/,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,2007-05-19,961000000,169.0,Released,"At the end of the world, the adventure begins.",6.9,4500
2,206647,Spectre,245000000,http://www.sonypictures.com/movies/spectre/,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,2015-10-26,880674609,148.0,Released,A Plan No One Escapes,6.3,4466
3,49026,The Dark Knight Rises,250000000,http://www.thedarkknightrises.com/,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,2012-07-16,1084939099,165.0,Released,The Legend Ends,7.6,9106
4,49529,John Carter,260000000,http://movies.disney.com/john-carter,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,2012-03-07,284139100,132.0,Released,"Lost in our world, found in another.",6.1,2124


In [8]:
def parse_json_column(df, movie_col, json_col):
    """
    Converts a JSON column into a normalized dataframe.
    Each JSON object becomes one row connected to movie_id.
    """
    rows = []

    for _, row in df.iterrows():
        movie_id = row[movie_col]
        data = json.loads(row[json_col])

        for item in data:
            item["movie_id"] = movie_id
            rows.append(item)

    return pd.DataFrame(rows)

In [9]:
# Movies JSON columns
genres_raw = parse_json_column(movies_df, "movie_id", "genres")
keywords_raw = parse_json_column(movies_df, "movie_id", "keywords")
companies_raw = parse_json_column(movies_df, "movie_id", "production_companies")
countries_raw = parse_json_column(movies_df, "movie_id", "production_countries")
languages_raw = parse_json_column(movies_df, "movie_id", "spoken_languages")

# Credits JSON columns
cast_raw = parse_json_column(credits_df, "movie_id", "cast")
crew_raw = parse_json_column(credits_df, "movie_id", "crew")

In [10]:
# Genre tables
genre_table = genres_raw[["id", "name"]].rename(
    columns={"id": "genre_id", "name": "genre_name"}
).drop_duplicates(subset=["genre_id"])

movie_genre_table = genres_raw[["movie_id", "id"]].rename(
    columns={"id": "genre_id"}
).drop_duplicates()

In [11]:
# Keyword tables
keyword_table = keywords_raw[["id", "name"]].rename(
    columns={"id": "keyword_id", "name": "keyword_name"}
).drop_duplicates(subset=["keyword_id"])

movie_keyword_table = keywords_raw[["movie_id", "id"]].rename(
    columns={"id": "keyword_id"}
).drop_duplicates()

In [12]:
# Production company tables
company_table = companies_raw[["id", "name"]].rename(
    columns={"id": "company_id", "name": "company_name"}
).drop_duplicates(subset=["company_id"])

movie_company_table = companies_raw[["movie_id", "id"]].rename(
    columns={"id": "company_id"}
).drop_duplicates()

In [13]:
# Production country tables
country_table = countries_raw[["iso_3166_1", "name"]].rename(
    columns={"iso_3166_1": "country_code", "name": "country_name"}
).drop_duplicates(subset=["country_code"])

movie_country_table = countries_raw[["movie_id", "iso_3166_1"]].rename(
    columns={"iso_3166_1": "country_code"}
).drop_duplicates()

In [14]:
# Spoken language tables
language_table = languages_raw[["iso_639_1", "name"]].rename(
    columns={"iso_639_1": "language_code", "name": "language_name"}
).drop_duplicates(subset=["language_code"])

movie_language_table = languages_raw[["movie_id", "iso_639_1"]].rename(
    columns={"iso_639_1": "language_code"}
).drop_duplicates()

In [15]:
credits_df = credits_df.rename(columns={"movie_id": "movie_id"})

cast_raw = parse_json_column(credits_df, "movie_id", "cast")
crew_raw = parse_json_column(credits_df, "movie_id", "crew")

In [16]:
# Actor tables
actor_table = cast_raw[["id", "name", "gender"]].rename(
    columns={"id": "actor_id", "name": "actor_name"}
).drop_duplicates(subset=["actor_id"])

movie_actor_table = cast_raw[["movie_id", "id", "character", "order"]].rename(
    columns={"id": "actor_id", "order": "cast_order"}
).drop_duplicates(subset=["movie_id", "actor_id", "character"])

In [17]:
# Crew tables
crew_table = crew_raw[["id", "name", "gender"]].rename(
    columns={"id": "crew_id", "name": "crew_name"}
).drop_duplicates(subset=["crew_id"])

movie_crew_table = crew_raw[["movie_id", "id", "department", "job"]].rename(
    columns={"id": "crew_id"}
).drop_duplicates(subset=["movie_id", "crew_id", "department", "job"])

Durante el proceso de transformación, convertí las columnas JSON en múltiples filas utilizando una función en Python. Cada elemento dentro del JSON se convirtió en una fila asociada a su `movie_id`.

Esto permitió pasar de una estructura no normalizada a tablas relacionales que pueden ser usadas en SQL.

## 3.2 Creación de la Base de Datos

En esta sección se crea una base de datos SQLite en el entorno de Kaggle para almacenar las tablas normalizadas.

In [18]:
db_path = "/kaggle/working/tmdb_normalized.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Enable foreign key support in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

In [19]:
cursor.executescript("""
DROP TABLE IF EXISTS Movie_Genre;
DROP TABLE IF EXISTS Movie_Keyword;
DROP TABLE IF EXISTS Movie_Production_Company;
DROP TABLE IF EXISTS Movie_Production_Country;
DROP TABLE IF EXISTS Movie_Spoken_Language;
DROP TABLE IF EXISTS Movie_Actor;
DROP TABLE IF EXISTS Movie_Crew;

DROP TABLE IF EXISTS Genre;
DROP TABLE IF EXISTS Keyword;
DROP TABLE IF EXISTS Production_Company;
DROP TABLE IF EXISTS Production_Country;
DROP TABLE IF EXISTS Spoken_Language;
DROP TABLE IF EXISTS Actor;
DROP TABLE IF EXISTS Crew_Member;
DROP TABLE IF EXISTS Movie;

CREATE TABLE Movie (
    movie_id INTEGER PRIMARY KEY,
    title TEXT,
    budget INTEGER,
    homepage TEXT,
    original_language TEXT,
    original_title TEXT,
    overview TEXT,
    popularity REAL,
    release_date TEXT,
    revenue INTEGER,
    runtime REAL,
    status TEXT,
    tagline TEXT,
    vote_average REAL,
    vote_count INTEGER
);

CREATE TABLE Genre (
    genre_id INTEGER PRIMARY KEY,
    genre_name TEXT
);

CREATE TABLE Keyword (
    keyword_id INTEGER PRIMARY KEY,
    keyword_name TEXT
);

CREATE TABLE Production_Company (
    company_id INTEGER PRIMARY KEY,
    company_name TEXT
);

CREATE TABLE Production_Country (
    country_code TEXT PRIMARY KEY,
    country_name TEXT
);

CREATE TABLE Spoken_Language (
    language_code TEXT PRIMARY KEY,
    language_name TEXT
);

CREATE TABLE Actor (
    actor_id INTEGER PRIMARY KEY,
    actor_name TEXT,
    gender INTEGER
);

CREATE TABLE Crew_Member (
    crew_id INTEGER PRIMARY KEY,
    crew_name TEXT,
    gender INTEGER
);

CREATE TABLE Movie_Genre (
    movie_id INTEGER,
    genre_id INTEGER,
    PRIMARY KEY (movie_id, genre_id),
    FOREIGN KEY (movie_id) REFERENCES Movie(movie_id),
    FOREIGN KEY (genre_id) REFERENCES Genre(genre_id)
);

CREATE TABLE Movie_Keyword (
    movie_id INTEGER,
    keyword_id INTEGER,
    PRIMARY KEY (movie_id, keyword_id),
    FOREIGN KEY (movie_id) REFERENCES Movie(movie_id),
    FOREIGN KEY (keyword_id) REFERENCES Keyword(keyword_id)
);

CREATE TABLE Movie_Production_Company (
    movie_id INTEGER,
    company_id INTEGER,
    PRIMARY KEY (movie_id, company_id),
    FOREIGN KEY (movie_id) REFERENCES Movie(movie_id),
    FOREIGN KEY (company_id) REFERENCES Production_Company(company_id)
);

CREATE TABLE Movie_Production_Country (
    movie_id INTEGER,
    country_code TEXT,
    PRIMARY KEY (movie_id, country_code),
    FOREIGN KEY (movie_id) REFERENCES Movie(movie_id),
    FOREIGN KEY (country_code) REFERENCES Production_Country(country_code)
);

CREATE TABLE Movie_Spoken_Language (
    movie_id INTEGER,
    language_code TEXT,
    PRIMARY KEY (movie_id, language_code),
    FOREIGN KEY (movie_id) REFERENCES Movie(movie_id),
    FOREIGN KEY (language_code) REFERENCES Spoken_Language(language_code)
);

CREATE TABLE Movie_Actor (
    movie_id INTEGER,
    actor_id INTEGER,
    character TEXT,
    cast_order INTEGER,
    PRIMARY KEY (movie_id, actor_id, character),
    FOREIGN KEY (movie_id) REFERENCES Movie(movie_id),
    FOREIGN KEY (actor_id) REFERENCES Actor(actor_id)
);

CREATE TABLE Movie_Crew (
    movie_id INTEGER,
    crew_id INTEGER,
    department TEXT,
    job TEXT,
    PRIMARY KEY (movie_id, crew_id, department, job),
    FOREIGN KEY (movie_id) REFERENCES Movie(movie_id),
    FOREIGN KEY (crew_id) REFERENCES Crew_Member(crew_id)
);
""")

conn.commit()

Se crearon tablas separadas para cada entidad (Actor, Genre, Keyword, etc.) y tablas intermedias para representar relaciones muchos-a-muchos.

Por ejemplo, una película puede tener múltiples actores, por lo que se utilizó la tabla `Movie_Actor` para representar esa relación.

## 3.3 Carga de Datos

Se insertan los datos procesados en las tablas de la base de datos utilizando pandas.

In [20]:
movie_table.to_sql("Movie", conn, if_exists="append", index=False)

genre_table.to_sql("Genre", conn, if_exists="append", index=False)
keyword_table.to_sql("Keyword", conn, if_exists="append", index=False)
company_table.to_sql("Production_Company", conn, if_exists="append", index=False)
country_table.to_sql("Production_Country", conn, if_exists="append", index=False)
language_table.to_sql("Spoken_Language", conn, if_exists="append", index=False)
actor_table.to_sql("Actor", conn, if_exists="append", index=False)
crew_table.to_sql("Crew_Member", conn, if_exists="append", index=False)

movie_genre_table.to_sql("Movie_Genre", conn, if_exists="append", index=False)
movie_keyword_table.to_sql("Movie_Keyword", conn, if_exists="append", index=False)
movie_company_table.to_sql("Movie_Production_Company", conn, if_exists="append", index=False)
movie_country_table.to_sql("Movie_Production_Country", conn, if_exists="append", index=False)
movie_language_table.to_sql("Movie_Spoken_Language", conn, if_exists="append", index=False)
movie_actor_table.to_sql("Movie_Actor", conn, if_exists="append", index=False)
movie_crew_table.to_sql("Movie_Crew", conn, if_exists="append", index=False)

conn.commit()

In [21]:
tables = pd.read_sql_query("""
SELECT name 
FROM sqlite_master 
WHERE type='table';
""", conn)

tables

,name
0,Movie
1,Genre
2,Keyword
3,Production_Company
4,Production_Country
5,Spoken_Language
6,Actor
7,Crew_Member
8,Movie_Genre
9,Movie_Keyword


In [22]:
for table in tables["name"]:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS rows FROM {table};", conn)
    print(table, count["rows"][0])

Movie 4803
Genre 20
Keyword 9813
Production_Company 5047
Production_Country 88
Spoken_Language 87
Actor 54588
Crew_Member 52885
Movie_Genre 12160
Movie_Keyword 36194
Movie_Production_Company 13677
Movie_Production_Country 6436
Movie_Spoken_Language 6937
Movie_Actor 106257
Movie_Crew 129581


Los conteos obtenidos tienen sentido considerando el dataset. Por ejemplo, existen muchas más filas en `Movie_Actor` que en `Movie`, lo cual es esperado porque cada película tiene varios actores.

Esto confirma que las relaciones muchos-a-muchos fueron correctamente implementadas.

In [23]:
pd.read_sql_query("""
SELECT 
    m.title,
    g.genre_name
FROM Movie m
JOIN Movie_Genre mg ON m.movie_id = mg.movie_id
JOIN Genre g ON mg.genre_id = g.genre_id
LIMIT 10;
""", conn)

,title,genre_name
0,Avatar,Action
1,Avatar,Adventure
2,Avatar,Fantasy
3,Avatar,Science Fiction
4,Pirates of the Caribbean: At World's End,Adventure
5,Pirates of the Caribbean: At World's End,Fantasy
6,Pirates of the Caribbean: At World's End,Action
7,Spectre,Action
8,Spectre,Adventure
9,Spectre,Crime


El resultado muestra que una misma película aparece en varias filas con diferentes géneros. Esto es correcto en un diseño normalizado, ya que cada fila representa una relación individual entre Movie y Genre.

Esto confirma que las llaves foráneas y las tablas intermedias funcionan correctamente.

In [24]:
conn.close()

print("Database created successfully:", db_path)

Database created successfully: /kaggle/working/tmdb_normalized.db


FASE 4: Consultas y Análisis de Datos
Realizar los siguientes scripts SQL 

• Búsqueda por Palabra Clave: El usuario ingresa una palabra y el sistema retorna
una tabla (máx. 15 filas) con: Título, Género, Lenguajes, Popularidad y Cast.

• Filtro Cronológico: El usuario ingresa un año. Mostrar películas estrenadas en ese
año o posteriores, ordenadas ascendentemente por año y luego alfabéticamente
por título (máx. 15 filas).

• Métricas por Género: Obtener una tabla que posea el nombre del genero, el
número de películas y el promedio de votos por género. Filtrar solo aquellos cuya
popularidad promedio sea mayor o igual a 6, ordenados de mayor a menor
popularidad (máx. 10 filas).

• Métricas por Año: Realizar el mismo análisis estadístico anterior, pero agrupado
por año de estreno.

In [25]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/kaggle/working/tmdb_normalized.db")

In [26]:
palabra = "avatar"

pd.read_sql_query("""
SELECT 
    m.title AS Titulo,
    GROUP_CONCAT(DISTINCT g.genre_name) AS Genero,
    GROUP_CONCAT(DISTINCT l.language_name) AS Lenguajes,
    m.popularity AS Popularidad,
    GROUP_CONCAT(DISTINCT a.actor_name) AS Cast
FROM Movie m
LEFT JOIN Movie_Genre mg ON m.movie_id = mg.movie_id
LEFT JOIN Genre g ON mg.genre_id = g.genre_id
LEFT JOIN Movie_Spoken_Language ml ON m.movie_id = ml.movie_id
LEFT JOIN Spoken_Language l ON ml.language_code = l.language_code
LEFT JOIN Movie_Actor ma ON m.movie_id = ma.movie_id
LEFT JOIN Actor a ON ma.actor_id = a.actor_id
WHERE m.title LIKE '%' || ? || '%'
   OR g.genre_name LIKE '%' || ? || '%'
   OR a.actor_name LIKE '%' || ? || '%'
GROUP BY m.movie_id
ORDER BY m.popularity DESC
LIMIT 15;
""", conn, params=(palabra, palabra, palabra))

,Titulo,Genero,Lenguajes,Popularidad,Cast
0,Avatar,"Adventure,Fantasy,Action,Science Fiction","English,Español",150.437577,"Giovanni Ribisi,Zoe Saldana,Sigourney Weaver,L..."


In [27]:
year = 2000

pd.read_sql_query("""
SELECT 
    title AS Titulo,
    release_date AS Fecha_Estreno,
    popularity AS Popularidad,
    vote_average AS Promedio_Votos
FROM Movie
WHERE CAST(substr(release_date, 1, 4) AS INTEGER) >= ?
ORDER BY CAST(substr(release_date, 1, 4) AS INTEGER) ASC, title ASC
LIMIT 15;
""", conn, params=(year,))

,Titulo,Fecha_Estreno,Popularidad,Promedio_Votos
0,102 Dalmatians,2000-10-07,9.895061,5.1
1,28 Days,2000-04-06,5.418921,6.0
2,3 Strikes,2000-03-01,1.433692,5.9
3,Aberdeen,2000-07-05,1.068819,7.0
4,All the Pretty Horses,2000-12-11,7.524671,5.8
5,Almost Famous,2000-09-15,21.547446,7.4
6,American Psycho,2000-04-13,45.310443,7.3
7,Amores perros,2000-06-16,23.281616,7.6
8,An Everlasting Piece,2000-12-22,0.068246,6.0
9,Anatomy,2000-02-03,5.025938,6.1


In [28]:
pd.read_sql_query("""
SELECT 
    g.genre_name AS Genero,
    COUNT(DISTINCT m.movie_id) AS Numero_Peliculas,
    ROUND(AVG(m.vote_average), 2) AS Promedio_Votos,
    ROUND(AVG(m.popularity), 2) AS Promedio_Popularidad
FROM Genre g
JOIN Movie_Genre mg ON g.genre_id = mg.genre_id
JOIN Movie m ON mg.movie_id = m.movie_id
GROUP BY g.genre_id, g.genre_name
HAVING AVG(m.popularity) >= 6
ORDER BY Promedio_Popularidad DESC
LIMIT 10;
""", conn)

,Genero,Numero_Peliculas,Promedio_Votos,Promedio_Popularidad
0,Adventure,790,6.16,39.27
1,Animation,234,6.34,38.81
2,Science Fiction,535,6.01,36.45
3,Fantasy,424,6.10,36.39
4,Action,1154,5.99,30.94
5,Family,513,6.03,27.83
6,Mystery,348,6.18,24.59
7,Thriller,1274,6.01,24.46
8,War,144,6.71,23.78
9,Crime,696,6.27,22.85


In [29]:
pd.read_sql_query("""
SELECT 
    CAST(substr(release_date, 1, 4) AS INTEGER) AS Year,
    COUNT(DISTINCT movie_id) AS Numero_Peliculas,
    ROUND(AVG(vote_average), 2) AS Promedio_Votos,
    ROUND(AVG(popularity), 2) AS Promedio_Popularidad
FROM Movie
WHERE release_date IS NOT NULL
GROUP BY Year
HAVING AVG(popularity) >= 6
ORDER BY Promedio_Popularidad DESC
LIMIT 10;
""", conn)

,Year,Numero_Peliculas,Promedio_Votos,Promedio_Popularidad
0,1975,6,7.30,47.49
1,1942,2,7.35,45.69
2,1939,3,7.67,42.89
3,1937,2,7.65,42.10
4,1957,2,7.95,41.18
5,1960,3,7.87,40.90
6,2014,238,5.58,37.48
7,2015,216,5.59,37.26
8,2016,104,5.83,37.13
9,1927,1,8.00,32.35


In [30]:
conn.close()